In [ ]:
import os
import random
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import models
from sklearn.metrics import confusion_matrix, f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PATCHES_DIR = "patches"            # куда сохраняем патчи
PATCHES_SPLIT = "patches_split"    # train/val/test
OUT_DIR = "model_out"
os.makedirs(OUT_DIR, exist_ok=True)

# Параметры модели/датасета
IMG = 64            # можно увеличить до 128 при наличии GPU памяти
PATCH_SIZE = 64
GRID_N = 19         # сетка для генерации патчей
THRESHOLD = 0.10    # overlap threshold для присвоения метки
MAX_EMPTY_PER_IMAGE = 20
GLOBAL_EMPTY_LIMIT = 250000

BATCH = 128 if torch.cuda.is_available() else 32
NUM_WORKERS = min(8, (os.cpu_count() or 4))
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-5
USE_FOCAL = False
FOCAL_GAMMA = 2.0
GRAD_CLIP = 1.0
PATIENCE_ES = 6
NUM_CLASSES = 3     # 0 empty, 1 black, 2 white

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "BATCH:", BATCH, "NUM_WORKERS:", NUM_WORKERS)


Device: cuda BATCH: 128 NUM_WORKERS: 2


In [ ]:
!pip install -q roboflow
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q onnx onnxruntime onnxruntime-tools albumentations tqdm

from roboflow import Roboflow
import os, sys, traceback

API_KEY = ""
if not API_KEY:
    raise RuntimeError("Set ROBOFLOW_API_KEY environment variable before running this cell.")

rf = Roboflow(api_key=API_KEY)

# Parameters: workspace, project, version, export format
WORKSPACE = "synthetic-data-3ol2y"
PROJECT = "go-positions"
VERSION = 6
EXPORT_FORMAT = "yolov4pytorch"   # change if you need yolov5/yolov8 etc.

try:
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)
    out_dir = version.download(EXPORT_FORMAT)
    # Roboflow returns an object or a path-like; handle both
    out_path = getattr(out_dir, "location", None) or str(out_dir)
    print("Dataset downloaded to:", out_path)
except Exception as e:
    print("Failed to download dataset from Roboflow:", e)
    traceback.print_exc()
    raise


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Go-Positions-6 in yolov4pytorch:: 100%|██████████| 2411/2411 [00:00<00:00, 4400.91it/s]

Dataset downloaded to: /content/Go-Positions-6


In [ ]:
import os

DATA_DIR = "/content/Go-Positions-6"
print("DATA_DIR:", DATA_DIR)
assert os.path.isdir(DATA_DIR), f"Папка {DATA_DIR} не найдена. Укажите правильный путь."

for root, dirs, files in os.walk(DATA_DIR):
    print("\nROOT:", root)
    print("  dirs:", dirs[:10])
    print("  files:", [f for f in files if f.endswith((".txt",".csv",".json",".xml"))][:20])
    break

# Найдём candidate файлов аннотаций и classes рекурсивно
candidates = {"annotations": [], "classes": []}
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith((".txt", ".csv", ".json", ".xml")):
            path = os.path.join(root, f)
            name = f.lower()
            if "annot" in name or "label" in name or "bbox" in name or name.startswith("train") or name.startswith("valid") or name.startswith("annotations"):
                candidates["annotations"].append(path)
            if "class" in name or "_classes" in name or "classes" in name:
                candidates["classes"].append(path)

print("\nFound annotation candidates (first 20):")
for p in candidates["annotations"][:20]:
    print("  ", p)
print("\nFound classes candidates (first 20):")
for p in candidates["classes"][:20]:
    print("  ", p)

# Если ничего не найдено, покажем все файлы изображений и структуру data_dir
if not candidates["annotations"]:
    imgs = []
    for root, dirs, files in os.walk(DATA_DIR):
        for f in files:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                imgs.append(os.path.join(root,f))
                if len(imgs) >= 20:
                    break
        if len(imgs) >= 20:
            break
    print("\nNo annotation files found in dataset. Example image files (up to 20):")
    for p in imgs[:20]:
        print("  ", p)


DATA_DIR: /content/Go-Positions-6

ROOT: /content/Go-Positions-6
  dirs: ['test', 'train', 'valid']
  files: ['README.roboflow.txt', 'README.dataset.txt']

Found annotation candidates (first 20):
   /content/Go-Positions-6/test/_annotations.txt
   /content/Go-Positions-6/train/_annotations.txt
   /content/Go-Positions-6/valid/_annotations.txt

Found classes candidates (first 20):
   /content/Go-Positions-6/test/_classes.txt
   /content/Go-Positions-6/train/_classes.txt
   /content/Go-Positions-6/valid/_classes.txt


In [ ]:
# B. Универсальный парсер, который ищет аннотации и classes внутри DATA_DIR и возвращает anns и mapping
import os, re
from collections import defaultdict

DATA_DIR = "/content/Go-Positions-6"
ANNOT_PATTERNS = ["_annotations.txt", "annotations.txt", "train.txt", "train/_annotations.txt", "labels.txt"]
CLASSES_PATTERNS = ["_classes.txt", "classes.txt", "train/_classes.txt"]

def find_file_recursive(base_dir, patterns):
    found = []
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            for pat in patterns:
                if f.lower() == pat.lower() or pat.lower() in f.lower():
                    found.append(os.path.join(root, f))
    return found

ann_files = find_file_recursive(DATA_DIR, ANNOT_PATTERNS)
class_files = find_file_recursive(DATA_DIR, CLASSES_PATTERNS)

print("Annotation files found:", ann_files)
print("Classes files found:", class_files)

def read_classes_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return [ln.strip() for ln in f if ln.strip()]

def autodetect_mapping_from_classes(classes_list):
    if not classes_list:
        return None
    lower = [c.lower() for c in classes_list]
    mapping = {}
    for i,name in enumerate(lower):
        if "grid" in name or "empty" in name or "background" in name:
            mapping[i] = 0
        elif "black" in name or "dark" in name:
            mapping[i] = 1
        elif "white" in name or "light" in name:
            mapping[i] = 2
    if not mapping:
        if len(lower) == 3:
            mapping = {0:1, 1:0, 2:2}
        else:
            mapping = {i:(1 if i==0 else 0) for i in range(len(lower))}
    return mapping

mapping_auto = None
rf_classes = None
if class_files:
    rf_classes = read_classes_file(class_files[0])
    mapping_auto = autodetect_mapping_from_classes(rf_classes)
    print("Read classes from:", class_files[0])
    print("rf_classes:", rf_classes)
    print("Auto mapping:", mapping_auto)
else:
    print("No classes file found inside dataset. mapping_auto = None")

# Парсер аннотаций
def parse_annotations_from_file(path):
    anns = defaultdict(list)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = re.split(r'\s+', ln)
            # common Roboflow export: image_path x_min y_min x_max y_max class
            if len(parts) >= 6:
                img = parts[0]
                try:
                    x1 = float(parts[1]); y1 = float(parts[2]); x2 = float(parts[3]); y2 = float(parts[4])
                    cls = int(float(parts[5]))
                    anns[img].append((x1,y1,x2,y2,cls))
                    continue
                except:
                    pass
            # fallback: comma separated
            if "," in ln:
                parts2 = [p.strip() for p in ln.split(",") if p.strip()]
                if len(parts2) >= 6:
                    img = parts2[0]
                    try:
                        x1 = float(parts2[1]); y1 = float(parts2[2]); x2 = float(parts2[3]); y2 = float(parts2[4])
                        cls = int(float(parts2[5]))
                        anns[img].append((x1,y1,x2,y2,cls))
                        continue
                    except:
                        pass
            # if line doesn't match, skip
    return anns

anns = {}
if ann_files:
    anns = parse_annotations_from_file(ann_files[0])
    print("Parsed annotations for", len(anns), "images from", ann_files[0])
else:
    raise FileNotFoundError("Аннотационный файл не найден внутри скачанного датасета. Проверьте DATA_DIR.")

# - anns : dict image_rel_path -> list of boxes (x1,y1,x2,y2,class_idx)
# - mapping_auto : roboflow_index -> our_label (0 empty,1 black,2 white) или None

Annotation files found: ['/content/Go-Positions-6/test/_annotations.txt', '/content/Go-Positions-6/test/_annotations.txt', '/content/Go-Positions-6/train/_annotations.txt', '/content/Go-Positions-6/train/_annotations.txt', '/content/Go-Positions-6/valid/_annotations.txt', '/content/Go-Positions-6/valid/_annotations.txt']
Classes files found: ['/content/Go-Positions-6/test/_classes.txt', '/content/Go-Positions-6/test/_classes.txt', '/content/Go-Positions-6/train/_classes.txt', '/content/Go-Positions-6/train/_classes.txt', '/content/Go-Positions-6/valid/_classes.txt', '/content/Go-Positions-6/valid/_classes.txt']
Read classes from: /content/Go-Positions-6/test/_classes.txt
rf_classes: ['blackStone', 'grid', 'whiteStone']
Auto mapping: {0: 1, 1: 0, 2: 2}
Parsed annotations for 0 images from /content/Go-Positions-6/test/_annotations.txt


In [ ]:
# Preview генерации патчей
from collections import Counter
from PIL import Image
import os

mapping = mapping_auto if mapping_auto is not None else {0:0,1:1,2:2}
print("Using mapping:", mapping)

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interW = max(0, xB - xA); interH = max(0, yB - yA)
    interArea = interW * interH
    boxAArea = max(0, (boxA[2]-boxA[0])) * max(0, (boxA[3]-boxA[1]))
    boxBArea = max(0, (boxB[2]-boxB[0])) * max(0, (boxB[3]-boxB[1]))
    union = boxAArea + boxBArea - interArea
    return interArea / union if union>0 else 0.0

def preview_counts(anns, mapping, images_root=DATA_DIR, grid_n=GRID_N, threshold=THRESHOLD, preview_n=200):
    counts = Counter()
    processed = 0
    for img_rel, boxes in list(anns.items())[:preview_n]:
        img_path = img_rel if os.path.isabs(img_rel) else os.path.join(images_root, img_rel)
        if not os.path.exists(img_path):
            base = os.path.basename(img_rel)
            found = None
            for ext in [".jpg",".png",".jpeg"]:
                cand = os.path.join(images_root, base + ext)
                if os.path.exists(cand):
                    found = cand; break
            if found:
                img_path = found
            else:
                continue
        im = Image.open(img_path).convert("RGB")
        W,H = im.size
        step_x = W / grid_n; step_y = H / grid_n
        for gy in range(grid_n):
            for gx in range(grid_n):
                x1 = int(round(gx * step_x)); y1 = int(round(gy * step_y))
                x2 = int(round(min(W, (gx+1) * step_x))); y2 = int(round(min(H, (gy+1) * step_y)))
                cell_box = (x1,y1,x2,y2)
                assigned_label = 0; best_iou = 0.0
                for b in boxes:
                    try:
                        bx1,by1,bx2,by2,cls_idx = b
                        box_abs = (int(round(bx1)), int(round(by1)), int(round(bx2)), int(round(by2)))
                    except:
                        continue
                    ov = iou(cell_box, box_abs)
                    if ov > best_iou:
                        best_iou = ov
                        if ov >= threshold:
                            assigned_label = mapping.get(int(cls_idx), 0)
                counts[assigned_label] += 1
        processed += 1
    return counts, processed

counts, processed = preview_counts(anns, mapping, images_root=DATA_DIR, grid_n=GRID_N, threshold=THRESHOLD, preview_n=200)
print("Preview processed images:", processed)
print("Preview patch counts:", dict(counts))


Using mapping: {0: 1, 1: 0, 2: 2}
Preview processed images: 0
Preview patch counts: {}


In [ ]:
import os, random, shutil
from collections import defaultdict, Counter
from PIL import Image

OUT_PATCHES = "patches"
SPLIT_DIR = "patches_split"
PATCH_SIZE = 64
GRID_N = 19
THRESHOLD = 0.10
MAX_EMPTY_PER_IMAGE = 20
GLOBAL_EMPTY_LIMIT = 250000
SEED = 42

# Проверки
assert 'anns' in globals(), "anns не найден. Выполните ячейку B и убедитесь, что anns загружен."
mapping = mapping_auto if ('mapping_auto' in globals() and mapping_auto is not None) else {0:0,1:1,2:2}
print("Using mapping:", mapping)

# Очистка/создание папки patches
os.makedirs(OUT_PATCHES, exist_ok=True)
for f in os.listdir(OUT_PATCHES):
    if f.endswith(".png"):
        os.remove(os.path.join(OUT_PATCHES, f))

global_empty = 0
counts = Counter()
saved = 0

for img_rel, boxes in list(anns.items()):
    img_path = img_rel if os.path.isabs(img_rel) else os.path.join(DATA_DIR, img_rel)
    if not os.path.exists(img_path):
        base = os.path.basename(img_rel)
        found = None
        for ext in [".jpg",".png",".jpeg"]:
            cand = os.path.join(DATA_DIR, base + ext)
            if os.path.exists(cand):
                found = cand; break
        if found:
            img_path = found
        else:
            continue
    try:
        im = Image.open(img_path).convert("RGB")
    except:
        continue
    W,H = im.size
    step_x = W / GRID_N; step_y = H / GRID_N
    basename = os.path.splitext(os.path.basename(img_path))[0]
    empty_per_image = 0
    idx = 0
    for gy in range(GRID_N):
        for gx in range(GRID_N):
            x1 = int(round(gx * step_x)); y1 = int(round(gy * step_y))
            x2 = int(round(min(W, (gx+1) * step_x))); y2 = int(round(min(H, (gy+1) * step_y)))
            cell_box = (x1,y1,x2,y2)
            assigned_label = 0; best_iou = 0.0
            for b in boxes:
                try:
                    bx1,by1,bx2,by2,cls_idx = b
                    box_abs = (int(round(bx1)), int(round(by1)), int(round(bx2)), int(round(by2)))
                except:
                    continue
                # IoU
                xa = max(cell_box[0], box_abs[0]); ya = max(cell_box[1], box_abs[1])
                xb = min(cell_box[2], box_abs[2]); yb = min(cell_box[3], box_abs[3])
                interW = max(0, xb - xa); interH = max(0, yb - ya)
                interA = interW * interH
                aA = max(0, (cell_box[2]-cell_box[0])) * max(0, (cell_box[3]-cell_box[1]))
                aB = max(0, (box_abs[2]-box_abs[0])) * max(0, (box_abs[3]-box_abs[1]))
                union = aA + aB - interA
                ov = interA / union if union>0 else 0.0
                if ov > best_iou:
                    best_iou = ov
                    if ov >= THRESHOLD:
                        assigned_label = mapping.get(int(cls_idx), 0)
            if assigned_label == 0:
                empty_per_image += 1
            if assigned_label == 0 and (empty_per_image > MAX_EMPTY_PER_IMAGE or global_empty >= GLOBAL_EMPTY_LIMIT):
                continue
            patch = im.crop((x1,y1,x2,y2)).resize((PATCH_SIZE, PATCH_SIZE))
            out_name = f"{assigned_label}_{basename}_{idx}.png"
            out_path = os.path.join(OUT_PATCHES, out_name)
            patch.save(out_path)
            counts[assigned_label] += 1
            saved += 1
            if assigned_label == 0:
                global_empty += 1
            idx += 1

print("Saved patches:", saved, "counts:", dict(counts))

# Stratified split by basename (groups)
groups = defaultdict(list)
for f in sorted(os.listdir(OUT_PATCHES)):
    if not f.endswith(".png"): continue
    parts = f[:-4].split("_")
    basename = "_".join(parts[1:-1]) if len(parts)>=3 else parts[1] if len(parts)>1 else parts[0]
    groups[basename].append(f)

group_label = {}
for b, files in groups.items():
    lbls = [int(f.split("_",1)[0]) for f in files]
    group_label[b] = Counter(lbls).most_common(1)[0][0]

by_label = defaultdict(list)
for b,lbl in group_label.items():
    by_label[lbl].append(b)

random.seed(SEED)
train_keys, val_keys, test_keys = [], [], []
for lbl, blist in by_label.items():
    random.shuffle(blist)
    n = len(blist)
    n_train = max(1, int(n * 0.8))
    n_val = max(1, int(n * 0.1)) if n>2 else 0
    train_keys.extend(blist[:n_train])
    val_keys.extend(blist[n_train:n_train+n_val])
    test_keys.extend(blist[n_train+n_val:])

if len(train_keys) == 0 and len(groups)>0:
    train_keys.append(next(iter(groups.keys())))

# write splits
for s in ["train","val","test"]:
    d = os.path.join(SPLIT_DIR, s)
    os.makedirs(d, exist_ok=True)
    # clear pngs
    for f in os.listdir(d):
        if f.endswith(".png"):
            os.remove(os.path.join(d, f))

def copy_group(keys, split):
    dst = os.path.join(SPLIT_DIR, split)
    for k in keys:
        for fname in groups[k]:
            shutil.copy2(os.path.join(OUT_PATCHES, fname), os.path.join(dst, fname))

copy_group(train_keys, "train")
copy_group(val_keys, "val")
copy_group(test_keys, "test")

print("Split done. Groups total:", len(groups), "train:", len(train_keys), "val:", len(val_keys), "test:", len(test_keys))
print("Files in splits:", {s: len(os.listdir(os.path.join(SPLIT_DIR, s))) for s in ['train','val','test']})


Using mapping: {0: 1, 1: 0, 2: 2}
Saved patches: 0 counts: {}
Split done. Groups total: 0 train: 0 val: 0 test: 0
Files in splits: {'train': 0, 'val': 0, 'test': 0}


In [ ]:
DATA_ROOT_CANDIDATES = ["/content", "."]
OUT_PATCHES = "patches"
SPLIT_DIR = "patches_split"
OUT_MODEL = "model_out"
os.makedirs(OUT_MODEL, exist_ok=True)

PATCH_SIZE = 64
GRID_N = 19
THRESHOLD = 0.10
MAX_EMPTY_PER_IMAGE = 20
GLOBAL_EMPTY_LIMIT = 250000
PREVIEW_N_IMAGES = 200

IMG = 64
BATCH = 128 if torch.cuda.is_available() else 32
NUM_WORKERS = 2
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-5
USE_FOCAL = False
FOCAL_GAMMA = 2.0
GRAD_CLIP = 1.0
PATIENCE_ES = 6
NUM_CLASSES = 3
SEED = 42

# ---------------- Imports and seed ----------------
import os, re, json, random, shutil, math, xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
import numpy as np
from tqdm import tqdm

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from sklearn.metrics import confusion_matrix, f1_score, classification_report

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "BATCH:", BATCH, "NUM_WORKERS:", NUM_WORKERS)

# ---------------- Helpers: find dataset dir and candidate files ----------------
def find_dataset_dir(candidates):
    for base in candidates:
        base = Path(base)
        if not base.exists(): continue
        for root, dirs, files in os.walk(base):
            files_l = [f.lower() for f in files]
            has_img = any(f.endswith((".jpg",".jpeg",".png")) for f in files_l)
            has_ann = any(("annot" in f or "annotation" in f or "labels" in f or f.endswith(".json") or f.endswith(".xml") or f.endswith(".txt")) for f in files_l)
            if has_img and has_ann:
                return Path(root)
    return None

def find_files_recursive(base, patterns):
    found = []
    for root, dirs, files in os.walk(base):
        for f in files:
            for pat in patterns:
                if f.lower() == pat.lower() or pat.lower() in f.lower():
                    found.append(os.path.join(root, f))
    return found

# ---------------- Detect dataset and files ----------------
DATA_DIR = find_dataset_dir(DATA_ROOT_CANDIDATES)
if DATA_DIR is None:
    if os.path.isdir("/content/Go-Positions-6"):
        DATA_DIR = Path("/content/Go-Positions-6")
if DATA_DIR is None:
    raise RuntimeError("Не найден скачанный датасет. Укажите правильную папку в DATA_ROOT_CANDIDATES.")
print("Found dataset dir:", DATA_DIR)

ann_candidates = find_files_recursive(DATA_DIR, ["_annotations.txt","annotations.txt","train.txt","labels.txt","_annotations.csv","_annotations.json"])
class_candidates = find_files_recursive(DATA_DIR, ["_classes.txt","classes.txt","_classes.csv","classes.csv"])
print("Annotation candidates:", ann_candidates[:5])
print("Classes candidates:", class_candidates[:5])

if not ann_candidates:
    raise RuntimeError("Аннотационный файл не найден в скачанном датасете. Убедитесь, что экспорт Roboflow был скачан.")

ANN_PATH = ann_candidates[0]
print("Using annotation file:", ANN_PATH)

def parse_annotations_roboflow_style(path):
    anns = defaultdict(list)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln: continue
            parts = re.split(r'\s+', ln)
            # find the token that looks like an image filename (contains .jpg/.png/.jpeg)
            img_token_idx = None
            for i, tok in enumerate(parts):
                if tok.lower().endswith((".jpg", ".jpeg", ".png")):
                    img_token_idx = i
                    break
            if img_token_idx is None:
                # fallback: if first token looks like filename without extension, skip
                continue
            img_rel = parts[img_token_idx]
            # remaining tokens after image token are bbox tokens
            bbox_tokens = parts[img_token_idx+1:]
            for bt in bbox_tokens:
                bt = bt.strip().strip(",")
                if not bt: continue
                coords = bt.split(",")
                if len(coords) < 5:
                    continue
                try:
                    x1 = float(coords[0]); y1 = float(coords[1]); x2 = float(coords[2]); y2 = float(coords[3])
                    cls = int(float(coords[4]))
                    anns[img_rel].append((x1,y1,x2,y2,cls))
                except:
                    continue
    return anns

anns = parse_annotations_roboflow_style(ANN_PATH)
print("Parsed annotations for", len(anns), "images (keys are image filenames as in annotation file).")
if len(anns) == 0:
    print("\n--- Diagnostic: first 40 lines of annotation file ---")
    with open(ANN_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for i,ln in enumerate(f):
            print(i+1, ln.strip())
            if i>=39: break
    raise RuntimeError("Не удалось распарсить аннотации. Пришлите первые строки файла аннотаций для диагностики.")

# ---------------- classes mapping auto-detect ----------------
rf_classes = None
mapping_auto = None
if class_candidates:
    try:
        with open(class_candidates[0], "r", encoding="utf-8", errors="ignore") as f:
            rf_classes = [ln.strip() for ln in f if ln.strip()]
    except:
        rf_classes = None
if rf_classes:
    lower = [c.lower() for c in rf_classes]
    mapping_auto = {}
    for i,name in enumerate(lower):
        if "grid" in name or "empty" in name or "background" in name:
            mapping_auto[i] = 0
        elif "black" in name or "dark" in name:
            mapping_auto[i] = 1
        elif "white" in name or "light" in name:
            mapping_auto[i] = 2
    if not mapping_auto and len(lower) == 3:
        mapping_auto = {0:1,1:0,2:2}
print("rf_classes:", rf_classes)
print("mapping_auto:", mapping_auto)

# default mapping if auto not found
mapping = mapping_auto if mapping_auto is not None else {0:0,1:1,2:2}
print("Using mapping (roboflow_index -> our_label):", mapping)

# ---------------- Preview (no write) to ensure we will get stone patches ----------------
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interW = max(0, xB - xA); interH = max(0, yB - yA)
    interArea = interW * interH
    boxAArea = max(0, (boxA[2]-boxA[0])) * max(0, (boxA[3]-boxA[1]))
    boxBArea = max(0, (boxB[2]-boxB[0])) * max(0, (boxB[3]-boxB[1]))
    union = boxAArea + boxBArea - interArea
    return interArea / union if union>0 else 0.0

print("\nRunning preview (no files will be written). This checks whether patch generation would produce labels 1/2.")
counts = Counter()
processed = 0
for img_rel, boxes in list(anns.items())[:PREVIEW_N_IMAGES]:
    img_path = img_rel if os.path.isabs(img_rel) else os.path.join(DATA_DIR, img_rel)
    if not os.path.exists(img_path):
        base = os.path.basename(img_rel)
        found = None
        for ext in [".jpg",".png",".jpeg"]:
            cand = os.path.join(DATA_DIR, base + ext)
            if os.path.exists(cand):
                found = cand; break
        if found:
            img_path = found
        else:
            continue
    try:
        im = Image.open(img_path).convert("RGB")
    except:
        continue
    W,H = im.size
    step_x = W / GRID_N; step_y = H / GRID_N
    for gy in range(GRID_N):
        for gx in range(GRID_N):
            x1 = int(round(gx * step_x)); y1 = int(round(gy * step_y))
            x2 = int(round(min(W, (gx+1) * step_x))); y2 = int(round(min(H, (gy+1) * step_y)))
            cell_box = (x1,y1,x2,y2)
            assigned_label = 0; best_iou = 0.0
            for b in boxes:
                try:
                    bx1,by1,bx2,by2,cls_idx = b
                    box_abs = (int(round(bx1)), int(round(by1)), int(round(bx2)), int(round(by2)))
                except:
                    continue
                ov = iou(cell_box, box_abs)
                if ov > best_iou:
                    best_iou = ov
                    if ov >= THRESHOLD:
                        assigned_label = mapping.get(int(cls_idx), 0)
            counts[assigned_label] += 1
    processed += 1

print("Preview processed images:", processed)
print("Preview patch counts:", dict(counts))
if counts.get(1,0) == 0 and counts.get(2,0) == 0:
    print("\nPreview did not find any patches labeled 1 or 2.")
    print("Possible fixes:")
    print(" - mapping may be wrong. rf_classes:", rf_classes)
    print(" - threshold may be too small/large; try THRESHOLD=0.06/0.2")
    print(" - grid size may be wrong; try GRID_N smaller (e.g., 9 or 13)")
    print("\nSample annotation keys (first 20):")
    for i,k in enumerate(list(anns.keys())[:20]):
        print(i+1, k, "->", anns[k][:3])
    raise RuntimeError("Preview found no stone labels. Adjust mapping/threshold/GRID_N or provide sample annotations for parser tuning.")

# ---------------- Full generation of patches ----------------
print("\nPreview OK — generating patches to", OUT_PATCHES)
os.makedirs(OUT_PATCHES, exist_ok=True)
# clear existing pngs
for f in os.listdir(OUT_PATCHES):
    if f.endswith(".png"):
        os.remove(os.path.join(OUT_PATCHES, f))

global_empty = 0
counts_full = Counter()
saved = 0
for img_rel, boxes in tqdm(anns.items(), desc="Generating patches"):
    img_path = img_rel if os.path.isabs(img_rel) else os.path.join(DATA_DIR, img_rel)
    if not os.path.exists(img_path):
        base = os.path.basename(img_rel)
        found = None
        for ext in [".jpg",".png",".jpeg"]:
            cand = os.path.join(DATA_DIR, base + ext)
            if os.path.exists(cand):
                found = cand; break
        if found:
            img_path = found
        else:
            continue
    try:
        im = Image.open(img_path).convert("RGB")
    except:
        continue
    W,H = im.size
    step_x = W / GRID_N; step_y = H / GRID_N
    basename = os.path.splitext(os.path.basename(img_path))[0]
    empty_per_image = 0
    idx = 0
    for gy in range(GRID_N):
        for gx in range(GRID_N):
            x1 = int(round(gx * step_x)); y1 = int(round(gy * step_y))
            x2 = int(round(min(W, (gx+1) * step_x))); y2 = int(round(min(H, (gy+1) * step_y)))
            cell_box = (x1,y1,x2,y2)
            assigned_label = 0; best_iou = 0.0
            for b in boxes:
                try:
                    bx1,by1,bx2,by2,cls_idx = b
                    box_abs = (int(round(bx1)), int(round(by1)), int(round(bx2)), int(round(by2)))
                except:
                    continue
                ov = iou(cell_box, box_abs)
                if ov > best_iou:
                    best_iou = ov
                    if ov >= THRESHOLD:
                        assigned_label = mapping.get(int(cls_idx), 0)
            if assigned_label == 0:
                empty_per_image += 1
            if assigned_label == 0 and (empty_per_image > MAX_EMPTY_PER_IMAGE or global_empty >= GLOBAL_EMPTY_LIMIT):
                continue
            patch = im.crop((x1,y1,x2,y2)).resize((PATCH_SIZE, PATCH_SIZE))
            out_name = f"{assigned_label}_{basename}_{idx}.png"
            out_path = os.path.join(OUT_PATCHES, out_name)
            patch.save(out_path)
            counts_full[assigned_label] += 1
            saved += 1
            if assigned_label == 0:
                global_empty += 1
            idx += 1

print("Saved patches:", saved, "counts:", dict(counts_full))

# ---------------- Stratified split by basename (groups) ----------------
print("\nCreating stratified split into:", SPLIT_DIR)
groups = defaultdict(list)
for f in sorted(os.listdir(OUT_PATCHES)):
    if not f.endswith(".png"): continue
    parts = f[:-4].split("_")
    basename = "_".join(parts[1:-1]) if len(parts)>=3 else parts[1] if len(parts)>1 else parts[0]
    groups[basename].append(f)

group_label = {}
for b, files in groups.items():
    lbls = [int(f.split("_",1)[0]) for f in files]
    group_label[b] = Counter(lbls).most_common(1)[0][0]

by_label = defaultdict(list)
for b,lbl in group_label.items():
    by_label[lbl].append(b)

random.seed(SEED)
train_keys, val_keys, test_keys = [], [], []
for lbl, blist in by_label.items():
    random.shuffle(blist)
    n = len(blist)
    n_train = max(1, int(n * 0.8))
    n_val = max(1, int(n * 0.1)) if n>2 else 0
    train_keys.extend(blist[:n_train])
    val_keys.extend(blist[n_train:n_train+n_val])
    test_keys.extend(blist[n_train+n_val:])

if len(train_keys) == 0 and len(groups)>0:
    train_keys.append(next(iter(groups.keys())))

# write splits
for s in ["train","val","test"]:
    d = os.path.join(SPLIT_DIR, s)
    os.makedirs(d, exist_ok=True)
    # clear pngs
    for f in os.listdir(d):
        if f.endswith(".png"):
            os.remove(os.path.join(d, f))

def copy_group(keys, split):
    dst = os.path.join(SPLIT_DIR, split)
    for k in keys:
        for fname in groups[k]:
            shutil.copy2(os.path.join(OUT_PATCHES, fname), os.path.join(dst, fname))

copy_group(train_keys, "train")
copy_group(val_keys, "val")
copy_group(test_keys, "test")

print("Split done. Groups total:", len(groups), "train groups:", len(train_keys), "val groups:", len(val_keys), "test groups:", len(test_keys))
print("Files in splits:", {s: len(os.listdir(os.path.join(SPLIT_DIR, s))) for s in ['train','val','test']})

# ---------------- Create datasets and dataloaders ----------------
print("\nRecreating datasets and dataloaders from split...")
USE_ALB = False
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    MEAN = (0.485,0.456,0.406); STD=(0.229,0.224,0.225)
    def make_rrc(sz):
        try:
            return A.RandomResizedCrop(height=sz, width=sz, scale=(0.9,1.0), p=0.5)
        except:
            try:
                return A.RandomResizedCrop(size=(sz,sz), scale=(0.9,1.0), p=0.5)
            except:
                return A.Compose([A.RandomCrop(height=int(sz*0.9), width=int(sz*0.9), p=0.5), A.Resize(height=sz, width=sz)])
    RRC = make_rrc(IMG)
    if RRC is None:
        raise Exception("RRC not available")
    train_aug = A.Compose([RRC, A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(p=0.5),
                           A.GaussNoise(p=0.2), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    val_aug = A.Compose([A.Resize(height=IMG, width=IMG), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    USE_ALB = True
except Exception:
    USE_ALB = False
    train_aug = None
    val_aug = None
    print("albumentations not available — using PIL->tensor fallback transforms")

FNAME_RE = re.compile(r'^(?P<label>\d+)_(?P<basename>.+)_(?P<idx>[^_]+)\.png$')
def parse_label(fname):
    m = FNAME_RE.match(os.path.basename(fname))
    if m:
        return int(m.group("label"))
    try:
        return int(os.path.basename(fname).split("_",1)[0])
    except:
        return 0

class PatchDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.folder = folder
        self.files = sorted([os.path.join(folder,f) for f in os.listdir(folder) if f.endswith(".png")])
        self.transform = transform
        self.labels = [parse_label(f) for f in self.files]
        self.counter = Counter(self.labels)
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        p = self.files[idx]
        img = Image.open(p).convert("RGB")
        img = np.array(img)
        if USE_ALB and self.transform is not None:
            res = self.transform(image=img); img_t = res['image']
        else:
            img_t = torch.from_numpy(img).permute(2,0,1).float() / 255.0
        lbl = parse_label(p)
        return img_t, lbl

train_ds = PatchDataset(os.path.join(SPLIT_DIR,"train"), transform=(train_aug if USE_ALB else None))
val_ds   = PatchDataset(os.path.join(SPLIT_DIR,"val"),   transform=(val_aug if USE_ALB else None))
test_ds  = PatchDataset(os.path.join(SPLIT_DIR,"test"),  transform=(val_aug if USE_ALB else None))

print("Dataset sizes (files):", len(train_ds), len(val_ds), len(test_ds))
print("Train label counts (dataset):", train_ds.counter)
print("Val label counts (dataset):", val_ds.counter)
print("Test label counts (dataset):", test_ds.counter)

if len(train_ds) == 0:
    raise RuntimeError("Train dataset empty after generation/split — проверьте аннотации/mapping/threshold.")

def make_sampler_and_weights(dataset, num_classes=NUM_CLASSES):
    counts = dataset.counter
    total = float(sum(counts.values())) if sum(counts.values())>0 else 1.0
    class_weights = [ total / (num_classes * counts.get(i, 1)) for i in range(num_classes) ]
    sample_weights = [ class_weights[parse_label(f)] for f in dataset.files ]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    return sampler, class_weights

sampler, class_weights_list = make_sampler_and_weights(train_ds)
print("Sampler class weights list:", class_weights_list)

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# ---------------- Model, loss, optimizer ----------------
def get_mobilenet_v2(num_classes=NUM_CLASSES, pretrained=True):
    try:
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
    except Exception:
        model = models.mobilenet_v2(pretrained=pretrained)
    in_features = model.last_channel if hasattr(model, "last_channel") else model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = get_mobilenet_v2(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)

train_counts = train_ds.counter
total = sum(train_counts.values()) if sum(train_counts.values())>0 else 1
class_weights = [ total / (NUM_CLASSES * train_counts.get(i, 1)) for i in range(NUM_CLASSES) ]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Class weights for loss:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) if not USE_FOCAL else None
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())

import torch.nn.functional as F
def focal_loss_fn(inputs, targets, alpha=None, gamma=2.0):
    ce = F.cross_entropy(inputs, targets, weight=alpha, reduction='none')
    p_t = torch.exp(-ce)
    loss = ((1 - p_t) ** gamma) * ce
    return loss.mean()

def evaluate(loader):
    model.eval()
    loss_sum = 0.0; total = 0; correct = 0
    all_preds=[]; all_labels=[]
    with torch.no_grad():
        for xb,yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
                out = model(xb)
                loss = (focal_loss_fn(out,yb,alpha=class_weights_tensor, gamma=FOCAL_GAMMA) if USE_FOCAL else criterion(out,yb))
            loss_sum += loss.item() * xb.size(0)
            preds = out.argmax(dim=1)
            correct += (preds==yb).sum().item()
            total += xb.size(0)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(yb.cpu().numpy().tolist())
    avg_loss = loss_sum/total if total else 0.0
    acc = correct/total if total else 0.0
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES))) if total else None
    return avg_loss, acc, cm, all_labels, all_preds

# ---------------- Training loop ----------------
best_mean_f1 = -1.0
best_path = os.path.join(OUT_MODEL, "best_model_by_mean_f1.pth")
no_improve = 0

print("\nStarting training...")
for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0; n_samples = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", ncols=120)
    for xb,yb in pbar:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
            out = model(xb)
            if USE_FOCAL:
                loss = focal_loss_fn(out,yb,alpha=class_weights_tensor, gamma=FOCAL_GAMMA)
            else:
                loss = criterion(out,yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        batch_n = xb.size(0)
        running_loss += loss.item() * batch_n
        n_samples += batch_n
        pbar.set_postfix(train_loss=running_loss / max(1, n_samples))
    train_loss = running_loss / max(1, n_samples)

    val_loss, val_acc, val_cm, val_labels, val_preds = evaluate(val_loader)
    f1_black = f1_score(val_labels, val_preds, labels=[1], average='macro') if 1 in set(val_labels) else 0.0
    f1_white = f1_score(val_labels, val_preds, labels=[2], average='macro') if 2 in set(val_labels) else 0.0
    mean_f1 = (f1_black + f1_white) / 2.0

    scheduler.step(mean_f1)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} mean_f1_stones={mean_f1:.4f} lr={current_lr:.6e}")
    if val_cm is not None:
        print("Val confusion matrix:\n", val_cm)

    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        torch.save(model.state_dict(), best_path)
        print("Saved best model by mean_f1:", best_mean_f1)
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE_ES:
        print("Early stopping — no improvement.")
        break

torch.save(model.state_dict(), os.path.join(OUT_MODEL, "final_model.pth"))
print("Training finished. Best mean_f1:", best_mean_f1)

# ---------------- Test evaluation ----------------
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.to(DEVICE).eval()
    test_loss, test_acc, test_cm, test_labels, test_preds = evaluate(test_loader)
    print("\nTest accuracy:", test_acc)
    print("Test confusion matrix:\n", test_cm)
    print("Classification report (test):")
    print(classification_report(test_labels, test_preds, labels=list(range(NUM_CLASSES)), target_names=["empty","black","white"]))
else:
    print("Best model not found, skipping test evaluation.")

print("\nPipeline finished. Artifacts: patches/, patches_split/, model_out/")


Device: cuda BATCH: 128 NUM_WORKERS: 2
Found dataset dir: /content/Go-Positions-6/test
Annotation candidates: ['/content/Go-Positions-6/test/_annotations.txt', '/content/Go-Positions-6/test/_annotations.txt']
Classes candidates: ['/content/Go-Positions-6/test/_classes.txt', '/content/Go-Positions-6/test/_classes.txt']
Using annotation file: /content/Go-Positions-6/test/_annotations.txt
Parsed annotations for 100 images (keys are image filenames as in annotation file).
rf_classes: ['blackStone', 'grid', 'whiteStone']
mapping_auto: {0: 1, 1: 0, 2: 2}
Using mapping (roboflow_index -> our_label): {0: 1, 1: 0, 2: 2}

Running preview (no files will be written). This checks whether patch generation would produce labels 1/2.
Preview processed images: 100
Preview patch counts: {0: 29787, 2: 3158, 1: 3155}

Preview OK — generating patches to patches


Generating patches: 100%|██████████| 100/100 [00:29<00:00,  3.40it/s]


Saved patches: 8313 counts: {0: 2000, 2: 3158, 1: 3155}

Creating stratified split into: patches_split
Split done. Groups total: 100 train groups: 79 val groups: 8 test groups: 13
Files in splits: {'train': 6490, 'val': 700, 'test': 1123}

Recreating datasets and dataloaders from split...
Dataset sizes (files): 6490 700 1123
Train label counts (dataset): Counter({1: 2457, 2: 2453, 0: 1580})
Val label counts (dataset): Counter({1: 271, 2: 269, 0: 160})
Test label counts (dataset): Counter({2: 436, 1: 427, 0: 260})
Sampler class weights list: [1.369198312236287, 0.8804775471442138, 0.8819133034379671]
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 134MB/s]


Class weights for loss: [1.369198312236287, 0.8804775471442138, 0.8819133034379671]

Starting training...


...

Epoch 18: train_loss=0.0872 val_loss=0.2115 val_acc=0.9086 mean_f1_stones=0.8815 lr=2.500000e-05
Val confusion matrix:
 [[160   0   0]
 [  0 241  30]
 [  0  34 235]]
Saved best model by mean_f1: 0.8814668477589827


Epoch 19/20: 100%|███████████████████████████████████████████████████| 51/51 [00:06<00:00,  8.24it/s, train_loss=0.0858]


Epoch 19: train_loss=0.0858 val_loss=0.2116 val_acc=0.9100 mean_f1_stones=0.8833 lr=2.500000e-05
Val confusion matrix:
 [[160   0   0]
 [  0 242  29]
 [  0  34 235]]
Saved best model by mean_f1: 0.8833137255574497


Epoch 20/20: 100%|███████████████████████████████████████████████████| 51/51 [00:04<00:00, 10.52it/s, train_loss=0.0881]


Epoch 20: train_loss=0.0881 val_loss=0.2180 val_acc=0.9086 mean_f1_stones=0.8815 lr=2.500000e-05
Val confusion matrix:
 [[160   0   0]
 [  0 240  31]
 [  0  33 236]]
Training finished. Best mean_f1: 0.8833137255574497

Test accuracy: 0.9073909171861086
Test confusion matrix:
 [[255   3   2]
 [  0 374  53]
 [  0  46 390]]
Classification report (test):
              precision    recall  f1-score   support

       empty       1.00      0.98      0.99       260
       black       0.88      0.88      0.88       427
       white       0.88      0.89      0.89       436

    accuracy                           0.91      1123
   macro avg       0.92      0.92      0.92      1123
weighted avg       0.91      0.91      0.91      1123


Pipeline finished. Artifacts: patches/, patches_split/, model_out/


In [ ]:
!pip install -q onnxscript onnx onnxruntime

model_cpu = model.to("cpu").eval()
dummy = torch.randn(1, 3, IMG, IMG)
torch.onnx.export(model_cpu, dummy, "model_out/model.onnx", input_names=["input"], output_names=["output"], opset_version=13)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.0 MB/s eta 0:00:00


W0620 14:15:09.202000 1883 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: adapt: Assertion `node-

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,64,64]>
            ),
            outputs=(
                %"output"<FLOAT,[1,3]>
            ),
            initializers=(
                %"features.0.0.weight"<FLOAT,[32,3,3,3]>{Tensor(...)},
                %"features.1.conv.0.0.weight"<FLOAT,[32,1,3,3]>{Tensor(...)},
                %"features.1.conv.1.weight"<FLOAT,[16,32,1,1]>{Tensor(...)},
                %"features.2.conv.1.0.weight"<FLOAT,[96,1,3,3]>{Tensor(...)},
                %"classifier.1.bias"<FLOAT,[3]>{TorchTensor<FLOAT,[3]>(Parameter containing: tensor([ 0.0067,  0.0154, -0.0250], requires_grad=True), name='classifier.1.bias')},
                %"features.2.conv.0.0.weight"